In [1]:
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>임베딩은 문장을 숫자 벡터로 바꾸고, 벡터스토어는 질문 벡터와 가까운 문서를 찾습니다. 세 문서만 색인해 의미가 가까운 결과가 먼저 나오는지 확인합니다.</p>

In [2]:
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>예제 실행</h3>
<p>서로 다른 주제의 짧은 문서 세 건과 문서 ID를 준비합니다.</p>

In [3]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

texts = [('D01', '파란도서관의 정기 휴관일은 매주 월요일입니다.'),
         ('D02', '해솔식당의 점심 영업은 오전 11시에 시작합니다.'),
         ('D03', '별빛회사의 급여일은 매월 25일입니다.')]
docs = [Document(page_content=t, metadata={'doc_id': i}) for i, t in texts]


<h3>실제 임베딩 검색</h3>
<p>OpenAI 임베딩으로 문서를 색인하고 도서관 질문과 가까운 결과를 출력합니다.</p>

In [4]:
embedding = OpenAIEmbeddings(model='text-embedding-3-small')
store = InMemoryVectorStore(embedding=embedding)
store.add_documents(docs)
hits = store.similarity_search_with_score('도서관이 쉬는 요일은?', k=2)
for doc, score in hits:
    print(doc.metadata['doc_id'], round(score, 3), doc.page_content)


D01 0.579 파란도서관의 정기 휴관일은 매주 월요일입니다.
D03 0.325 별빛회사의 급여일은 매월 25일입니다.


<h3>변형 실행</h3>
<p>질문을 급여일 질문으로 바꾼 뒤 첫 번째 문서 ID를 확인합니다.</p>

In [5]:
pay_hits = store.similarity_search('월급은 며칠에 받나요?', k=1)
print(pay_hits[0].metadata['doc_id'])


D03


<h3>일부러 실패해 보기</h3>
<p><code>k</code>에는 정수가 필요합니다. 문자열을 넘겨 실제 타입 오류를 포착합니다.</p>

In [6]:
try:
    store.similarity_search('휴관일', k='two')
except TypeError as error:
    print('예상한 실패:', error)


예상한 실패: slice indices must be integers or None or have an __index__ method


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [7]:
assert hits[0][0].metadata['doc_id'] == 'D01'
assert pay_hits[0].metadata['doc_id'] == 'D03'
print('검증 통과: 두 질문이 알맞은 문서를 검색했습니다.')


검증 통과: 두 질문이 알맞은 문서를 검색했습니다.


<h3>정리</h3>
<p>임베딩 모델은 단어가 정확히 같지 않아도 의미가 가까운 문서를 찾습니다. 검색 점수는 모델과 벡터스토어에 따라 해석이 달라지므로 같은 설정 안에서 순위를 비교합니다.</p>